In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

print("OpenCV:", cv2.__version__)
print("YOLO imported successfully")

In [ ]:
ROOT = Path(r"D:\CrowdTrackAI")

SEQUENCE = (
    ROOT
    / "data"
    / "raw"
    / "MOT17"
    / "train"
    / "MOT17-02-FRCNN"
)

FRAME_NUMBER = 100

IMAGE_PATH = SEQUENCE / "img1" / f"{FRAME_NUMBER:06d}.jpg"

assert IMAGE_PATH.exists(), f"Image not found: {IMAGE_PATH}"

# Pretrained YOLO detector
model = YOLO("yolo11n.pt")

print("Selected image:", IMAGE_PATH)
print("Detector loaded successfully")

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

if image is None:
    raise RuntimeError("Could not read the image")

# COCO class 0 represents a person.
results = model.predict(
    source=image,
    classes=[0],
    conf=0.25,
    verbose=False
)

result = results[0]

print("Detections:", len(result.boxes))

In [ ]:
annotated = result.plot()

annotated_rgb = cv2.cvtColor(
    annotated,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(16, 9))
plt.imshow(annotated_rgb)
plt.title(f"YOLO Person Detection — Frame {FRAME_NUMBER}")
plt.axis("off")
plt.show()

In [ ]:
import pandas as pd

boxes = result.boxes

if boxes is None or len(boxes) == 0:
    detections_df = pd.DataFrame(
        columns=["x1", "y1", "x2", "y2", "confidence"]
    )
else:
    coordinates = boxes.xyxy.cpu().numpy()
    confidence = boxes.conf.cpu().numpy()

    detections_df = pd.DataFrame(
        coordinates,
        columns=["x1", "y1", "x2", "y2"]
    )

    detections_df["confidence"] = confidence

display(detections_df)